# 特定動画の再生数推移 + 月次ランキング

旧06/10の統合版。日次CSV群から combined_df を構築し、
月次ランキング・特定動画の推移・直近3日上昇率を表示する。

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 設定
CHANNEL = "hima72"  #@param ["hima72", "sixfonia", "kosame", "illuma", "mikoto", "suchi", "lan"]
VIDEO_ID = "Qfp9NXawFZw"  #@param {type:"string"}
TARGET_VIEWS = 1000000  #@param {type:"integer"}
YEARS = "2025,2026"  #@param {type:"string"}

years = [y.strip() for y in YEARS.split(",") if y.strip()]

In [ ]:
#@title 📥 combined_df 構築
from sixfonia_analytics import load

combined_df = load.build_combined_df(CHANNEL, years=years)
combined_df.head()

In [ ]:
#@title 🏆 月次ランキング（直前の完全月 / 再生・いいね・コメント増加 Top10）
from sixfonia_analytics import auth, enrich, metrics, display as disp

monthly_gains = metrics.aggregate_monthly_gains(combined_df)

# タイトル付与（APIキー未設定でも videoId 表示で動く）
try:
    youtube = auth.build_youtube()
    titles = enrich.fetch_video_titles(youtube, monthly_gains["videoId"].tolist())
    monthly_gains["video_title"] = (
        monthly_gains["videoId"].map(titles).fillna(monthly_gains["videoId"])
    )
    print(f"タイトル取得: {len(titles)} 件")
except Exception as e:
    print(f"[INFO] タイトル取得スキップ: {e}")

for metric, label in [
    ("monthly_views_gain", "月間再生数増加 Top 10"),
    ("monthly_likes_gain", "月間いいね増加 Top 10"),
    ("monthly_comments_gain", "月間コメント増加 Top 10"),
]:
    disp.display_ranking_cards(monthly_gains, metric, label)

In [ ]:
#@title 🎯 特定動画の推移（テーブル + サムネ + CSV保存）
from IPython.display import HTML, display
from sixfonia_analytics import config, metrics

video_df = metrics.filter_video(combined_df, VIDEO_ID, target_views=TARGET_VIEWS)

if not video_df.empty:
    display(HTML(
        f"<p><b>Video ID:</b> {VIDEO_ID}</p>"
        f"<img src='{video_df.iloc[0]['thumbnail_url']}' width='480'>"
    ))
    display(video_df[["view_date", "videoId", "viewCount", "likeCount",
                      "commentCount", "viewCountDiff", "views_to_target"]])
    out = config.channel_dir(CHANNEL) / f"video_data_{VIDEO_ID}.csv"
    video_df.to_csv(out, index=False)
    print(f"保存: {out}")
else:
    print(f"[WARN] {VIDEO_ID} のデータがありません")

In [ ]:
#@title 📈 推移グラフ（累積 + 日次の2段）
from sixfonia_analytics import plots

plots.plot_view_trend(video_df, VIDEO_ID)

In [ ]:
#@title 📊 直近3日間の上昇率ランキング
from sixfonia_analytics import metrics, display as disp

recent_gains_df = metrics.get_recent_daily_gains(combined_df, n_days=3)
disp.display_recent_gain_ranking(recent_gains_df, titles_df=monthly_gains, top_n=10)